## Q1 — Real-World Impact Framing

- Mission domain: Earth / forest fire risk prediction for Algeria.
- Problem: predict whether a given day and region will experience a forest fire outbreak using weather and FWI measurements.
- Beneficiaries: forestry departments, fire services, rural communities, farmers, and emergency planners.
- Prediction target: binary label {fire, not fire} for a given day and location.
- Real-world impact: earlier warnings, faster deployment of crews and equipment, reduced burned area, lower economic and humanitarian losses.
- Why ML fits: fire danger is driven by nonlinear interactions among temperature, humidity, wind, rainfall, and FWI indices, which tree-based and ensemble models capture better than brittle rule thresholds.
- Source: Algerian Forest Fires Dataset from the UCI ML Repository, 244 daily records from June–September 2012 across two regions.
- One row represents one region-day.
- Responsible-use limits: small sample, one season, two regions only; weather-only model ignores fuels, terrain, human ignition, and future-year drift; false negatives are more costly than false positives, so recall should be prioritized in deployment.

## Q2 — Data Wrangling & Feature Engineering

This notebook builds a leakage-safe pipeline: handle missing values and invalid data, normalize labels, audit duplicates and outliers, encode region, engineer domain features, scale numeric variables, and keep the train/validation/test split reproducible. The feature engineering uses domain knowledge such as peak fire month, dryness index, and wind-dryness interaction to expose realistic fire-risk patterns before modelling.


In [3]:
def load_raw(path: Path = RAW_PATH) -> pd.DataFrame:
    """
    Load the ORIGINAL raw UCI CSV. This file ships as two stacked
    per-region tables (Bejaia rows 1-122, Sidi-Bel Abbes rows 123-244),
    each with its own repeated header + a free-text title row above it.
    That's why we can't just pd.read_csv() naively - header=1 skips the
    'Bejaia Region Dataset' title line, and we still have a second,
    embedded title+header row before the Sidi-Bel Abbes block that needs
    detecting and removing.
    """
    if not path.exists():
        raise FileNotFoundError(
            f"Dataset file not found at {path}. "
            "Please ensure the CSV is present or use the UCI API loader."
        )

    df = pd.read_csv(path, header=1, skip_blank_lines=True)

    # Strip stray whitespace from column names (raw file has ' RH', ' Ws',
    # 'Rain ', 'Classes ' etc. with leading/trailing spaces)
    df.columns = [c.strip() for c in df.columns]

    # The embedded "Sidi-Bel Abbes Region Dataset" title row, and the
    # header row repeated right after it, get read in as DATA rows
    # (their 'day' cell holds text, not a day-number) -> find & drop them.
    bad_mask = pd.to_numeric(df["day"], errors="coerce").isna()
    n_bad = bad_mask.sum()
    print(f"[load_raw] Found {n_bad} embedded title/header rows -> dropping them.")
    df = df.loc[~bad_mask].reset_index(drop=True)

    # Tag Region from position: first 122 valid rows = Bejaia,
    # remaining = Sidi-Bel Abbes (per official dataset documentation).
    df["Region"] = np.where(df.index < 122, "Bejaia", "Sidi-Bel Abbes")

    return df


def try_load_via_ucimlrepo() -> pd.DataFrame | None:
    """
    Official UCI loading code for this dataset (id=547), exactly as given
    in the assignment. This is the PRIMARY loader; load_raw() above is
    only a fallback for offline environments.
    """
    try:
        from ucimlrepo import fetch_ucirepo

        # fetch dataset
        algerian_forest_fires = fetch_ucirepo(id=547)

        # data (as pandas dataframes)
        X = algerian_forest_fires.data.features
        y = algerian_forest_fires.data.targets

        # metadata
        print(algerian_forest_fires.metadata)

        # variable information
        print(algerian_forest_fires.variables)

        df = pd.concat([X, y], axis=1)
        # The API can still ship column names with stray leading/trailing
        # whitespace (e.g. ' RH', 'Rain ') - normalize them so every
        # downstream column reference (numeric_cols, scale_cols, etc.)
        # works regardless of loader path.
        df.columns = [c.strip() for c in df.columns]

        # The API's features table doesn't include a Region column even
        # though the dataset is documented as 122 Bejaia rows followed by
        # 122 Sidi-Bel Abbes rows - reconstruct it the same way load_raw()
        # does, so both loading paths produce an identical schema.
        if "Region" not in df.columns:
            df = df.reset_index(drop=True)
            df["Region"] = np.where(df.index < 122, "Bejaia", "Sidi-Bel Abbes")

        print("\n[load] Loaded via ucimlrepo (id=547).")
        return df
    except Exception as e:
        print(f"[load] ucimlrepo unavailable ({e}); falling back to local raw CSV.")
        return None


df = try_load_via_ucimlrepo()
if df is None:
    if RAW_PATH.exists():
        df = load_raw()
    else:
        raise FileNotFoundError(
            f"No dataset found. Download the CSV to {RAW_PATH} or install ucimlrepo."
        )

print("\nShape after load:", df.shape)
print(df.dtypes)


{'uci_id': 547, 'name': 'Algerian Forest Fires', 'repository_url': 'https://archive.ics.uci.edu/dataset/547/algerian+forest+fires+dataset', 'data_url': 'https://archive.ics.uci.edu/static/public/547/data.csv', 'abstract': 'The dataset includes 244 instances that regroup a data of two regions of Algeria.', 'area': 'Biology', 'tasks': ['Classification', 'Regression'], 'characteristics': ['Multivariate'], 'num_instances': 244, 'num_features': 14, 'feature_types': ['Real'], 'demographics': [], 'target_col': ['Classes  '], 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 2019, 'last_updated': 'Tue Mar 19 2024', 'dataset_doi': '10.24432/C5KW4N', 'creators': [' Faroudja Abid'], 'intro_paper': {'ID': 325, 'type': 'NATIVE', 'title': ' Predicting Forest Fire in Algeria Using Data Mining Techniques: Case Study of the Decision Tree Algorithm', 'authors': 'Faroudja Abid, N.Izeboudjen', 'venue': 'Ezziyyani M. (eds) Advanced Intelligent Systems

In [ ]:
try:
    from ucimlrepo import fetch_ucirepo
    print("ucimlrepo is available.")
except Exception as e:
    print(f"ucimlrepo is not installed or unavailable: {e}")
    print("The main workflow includes a local CSV fallback so the notebook can still run.")


{'uci_id': 547, 'name': 'Algerian Forest Fires', 'repository_url': 'https://archive.ics.uci.edu/dataset/547/algerian+forest+fires+dataset', 'data_url': 'https://archive.ics.uci.edu/static/public/547/data.csv', 'abstract': 'The dataset includes 244 instances that regroup a data of two regions of Algeria.', 'area': 'Biology', 'tasks': ['Classification', 'Regression'], 'characteristics': ['Multivariate'], 'num_instances': 244, 'num_features': 14, 'feature_types': ['Real'], 'demographics': [], 'target_col': ['Classes  '], 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 2019, 'last_updated': 'Tue Mar 19 2024', 'dataset_doi': '10.24432/C5KW4N', 'creators': [' Faroudja Abid'], 'intro_paper': {'ID': 325, 'type': 'NATIVE', 'title': ' Predicting Forest Fire in Algeria Using Data Mining Techniques: Case Study of the Decision Tree Algorithm', 'authors': 'Faroudja Abid, N.Izeboudjen', 'venue': 'Ezziyyani M. (eds) Advanced Intelligent Systems

## Q4 — Explainability & Ethics

- Use SHAP or LIME on the best-performing model to explain both global feature importance and individual predictions.
- Interpret feature impact in plain language: higher temperature, lower humidity, little or no rain, and higher FWI-type indices generally increase fire risk.
- Discuss fairness and bias: the model is trained on a narrow climate/time window and may not generalize to other regions or future years without retraining.
- Privacy: this dataset is public and weather-only, so privacy risk is low; however, operational use should still respect local data governance and human oversight.
- Uncertainty: predictions should be treated as decision support, not autonomous decisions.
- Cost asymmetry: false negatives are costlier than false positives in wildfire management, so threshold tuning and recall-focused review are important.
- Human oversight: the model should support fire-watch decisions and emergency planning rather than replace expert judgment.

### Final project summary

This project addresses a real public-good problem by building a leakage-safe machine learning pipeline for forest fire risk prediction, comparing several ensemble models, and explaining the key drivers of model decisions in domain terms. The strongest model should be benchmarked against a simple baseline and used as decision support within a human-in-the-loop operational workflow.


In [ ]:
"""
CIA-3 ML for Social Good | Mission Earth - Forest Fire Risk Prediction
=======================================================================
STAGE: Q3 - Ensemble Architecture, Tuning, Comparison
       Q4 - Model Explainability & Ethics

Loads the leakage-safe splits produced by
01_data_cleaning_feature_engineering.py (in outputs/), trains a baseline
model plus bagging / boosting / stacking ensembles, compares them all on
the SAME untouched test set, then explains the best model with SHAP.
"""

import numpy as np
import pandas as pd
from pathlib import Path

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

OUT_DIR = Path("outputs")
OUT_DIR.mkdir(exist_ok=True)

# =========================================================================
# 1. LOAD THE Q2 SPLITS
# =========================================================================
X_train = pd.read_csv(OUT_DIR / "X_train.csv")
X_val = pd.read_csv(OUT_DIR / "X_val.csv")
X_test = pd.read_csv(OUT_DIR / "X_test.csv")
y_train = pd.read_csv(OUT_DIR / "y_train.csv").squeeze("columns")
y_val = pd.read_csv(OUT_DIR / "y_val.csv").squeeze("columns")
y_test = pd.read_csv(OUT_DIR / "y_test.csv").squeeze("columns")
X_train_bal = pd.read_csv(OUT_DIR / "X_train_smote.csv")
y_train_bal = pd.read_csv(OUT_DIR / "y_train_smote.csv").squeeze("columns")

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")
print(f"SMOTE-balanced train: {X_train_bal.shape}")


# =========================================================================
# 2. MODELS
#    - Baseline        : Logistic Regression (simple, linear, fast)
#    - Bagging          : Random Forest
#    - Boosting         : XGBoost (falls back to GradientBoosting if the
#                         xgboost package isn't installed on the grading
#                         machine, so the script never hard-fails)
#    - Stacking         : heterogeneous stack of RF + XGB/GB with a
#                         Logistic Regression meta-learner, fit with
#                         internal cross-validation so the meta-learner
#                         never sees the same rows it was trained on
#                         (this is what "prevent leakage in meta-learning"
#                         means in the rubric).
# =========================================================================
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (RandomForestClassifier, StackingClassifier,
                               GradientBoostingClassifier)
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (roc_auc_score, f1_score, precision_score,
                              recall_score, confusion_matrix,
                              classification_report)

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("xgboost not installed -> boosting model will use "
          "GradientBoostingClassifier instead.")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# --- Baseline: Logistic Regression, light tuning on C -------------------
baseline_grid = GridSearchCV(
    LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    param_grid={"C": [0.01, 0.1, 1, 10]},
    scoring="roc_auc", cv=cv, n_jobs=-1
)
baseline_grid.fit(X_train, y_train)
baseline_model = baseline_grid.best_estimator_
print(f"\n[Baseline] best C = {baseline_grid.best_params_}")

# --- Bagging: Random Forest, tune depth/estimators -----------------------
rf_grid = GridSearchCV(
    RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE),
    param_grid={
        "n_estimators": [100, 300],
        "max_depth": [3, 5, None],
        "min_samples_leaf": [1, 3],
    },
    scoring="roc_auc", cv=cv, n_jobs=-1
)
rf_grid.fit(X_train, y_train)
rf_model = rf_grid.best_estimator_
print(f"[Bagging - RF] best params = {rf_grid.best_params_}")

# --- Boosting: XGBoost (or GradientBoosting fallback) ---------------------
if HAS_XGB:
    boost_grid = GridSearchCV(
        XGBClassifier(eval_metric="logloss", random_state=RANDOM_STATE),
        param_grid={
            "n_estimators": [100, 300],
            "max_depth": [2, 3, 4],
            "learning_rate": [0.05, 0.1],
        },
        scoring="roc_auc", cv=cv, n_jobs=-1
    )
else:
    boost_grid = GridSearchCV(
        GradientBoostingClassifier(random_state=RANDOM_STATE),
        param_grid={
            "n_estimators": [100, 300],
            "max_depth": [2, 3],
            "learning_rate": [0.05, 0.1],
        },
        scoring="roc_auc", cv=cv, n_jobs=-1
    )
boost_grid.fit(X_train, y_train)
boost_model = boost_grid.best_estimator_
print(f"[Boosting] best params = {boost_grid.best_params_}")

# --- Stacking: heterogeneous RF + Boosting -> Logistic Regression meta ---
# StackingClassifier internally does k-fold cross-validation to generate
# out-of-fold predictions for the meta-learner, so the meta-learner is
# never trained on a base model's in-sample (leaked) predictions.
stack_model = StackingClassifier(
    estimators=[("rf", rf_model), ("boost", boost_model)],
    final_estimator=LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    cv=cv, n_jobs=-1
)
stack_model.fit(X_train, y_train)
print("[Stacking] fit complete.")


# =========================================================================
# 3. COMPARE ALL MODELS ON THE SAME UNTOUCHED TEST SET
# =========================================================================
models = {
    "Baseline (Logistic Regression)": baseline_model,
    "Bagging (Random Forest)": rf_model,
    "Boosting (XGBoost)" if HAS_XGB else "Boosting (GradientBoosting)": boost_model,
    "Stacking (RF + Boosting -> LR)": stack_model,
}

results = []
for name, model in models.items():
    proba = model.predict_proba(X_test)[:, 1]
    pred = model.predict(X_test)
    results.append({
        "Model": name,
        "ROC-AUC": roc_auc_score(y_test, proba),
        "F1": f1_score(y_test, pred),
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
    })

results_df = pd.DataFrame(results).sort_values("ROC-AUC", ascending=False)
print("\n" + "=" * 74)
print("MODEL COMPARISON ON HELD-OUT TEST SET")
print("=" * 74)
print(results_df.round(3).to_string(index=False))
results_df.to_csv(OUT_DIR / "model_comparison.csv", index=False)

best_name = results_df.iloc[0]["Model"]
best_model = models[best_name]
print(f"\nBest model by ROC-AUC: {best_name}")

baseline_auc = results_df.loc[
    results_df["Model"].str.contains("Baseline"), "ROC-AUC"
].values[0]
best_auc = results_df.iloc[0]["ROC-AUC"]
print(
    f"\nBest ensemble {'BEATS' if best_auc > baseline_auc else 'does NOT beat'} "
    f"the baseline: {best_auc:.3f} vs {baseline_auc:.3f} "
    f"({'+' if best_auc > baseline_auc else ''}{(best_auc - baseline_auc):.3f} ROC-AUC)."
)

# --- Confusion matrix for the best model ----------------------------------
best_pred = best_model.predict(X_test)
cm = confusion_matrix(y_test, best_pred)
print(f"\nConfusion matrix for {best_name} (rows=actual, cols=predicted; "
      f"0=not fire, 1=fire):")
print(pd.DataFrame(cm, index=["actual: not fire", "actual: fire"],
                    columns=["pred: not fire", "pred: fire"]))
print("\n" + classification_report(y_test, best_pred,
                                    target_names=["not fire", "fire"]))


# =========================================================================
# 4. Q4 - EXPLAINABILITY: SHAP on the best model (global + local)
# =========================================================================
import shap
import matplotlib
matplotlib.use("Agg")  # headless-safe for scripts/servers
import matplotlib.pyplot as plt

print("\nComputing SHAP values for the best model (this may take a moment)...")

# TreeExplainer works directly for RF/XGB/GradientBoosting. If the best
# model is the stacking ensemble (not a single tree model), explain its
# strongest base learner instead - SHAP's TreeExplainer doesn't support
# stacked meta-estimators directly, and this is a common, defensible
# workaround to note in your ethics/limitations discussion.
explain_model = best_model
explain_name = best_name
if isinstance(best_model, StackingClassifier):
    explain_model = boost_model
    explain_name = f"{best_name}'s boosting base-learner (stack itself isn't tree-SHAP-compatible)"
    print(f"Note: explaining {explain_name} instead of the stack directly.")

explainer = shap.TreeExplainer(explain_model)
shap_values = explainer.shap_values(X_test)
# Handle both the (n_samples, n_features) and (n_samples, n_features, 2)
# return shapes seen across sklearn/xgboost versions.
if isinstance(shap_values, list):
    shap_values_fire = shap_values[1]
elif shap_values.ndim == 3:
    shap_values_fire = shap_values[:, :, 1]
else:
    shap_values_fire = shap_values

# --- Global explanation: which features matter most overall --------------
plt.figure()
shap.summary_plot(shap_values_fire, X_test, show=False, plot_type="bar")
plt.title(f"Global feature importance - {explain_name}")
plt.tight_layout()
plt.savefig(OUT_DIR / "shap_global_importance.png", dpi=150)
plt.close()
print(f"Saved global SHAP plot -> {OUT_DIR / 'shap_global_importance.png'}")

mean_abs_shap = pd.Series(
    np.abs(shap_values_fire).mean(axis=0), index=X_test.columns
).sort_values(ascending=False)
print("\n--- Top 5 globally important features (mean |SHAP value|) ---")
print(mean_abs_shap.head(5).round(3))

# --- Local explanation: one individual prediction, explained -------------
# Pick the test-set row the model is MOST confident is a fire day, so the
# local explanation is easy to narrate in the pitch video.
best_proba = explain_model.predict_proba(X_test)[:, 1]
idx = int(np.argmax(best_proba))
row = X_test.iloc[[idx]]
print(f"\n--- Local explanation for test row #{idx} "
      f"(model predicts fire probability = {best_proba[idx]:.3f}) ---")
print(row.T)

plt.figure()
shap.plots._waterfall.waterfall_legacy(
    explainer.expected_value if not isinstance(explainer.expected_value, (list, np.ndarray))
    else explainer.expected_value[1],
    shap_values_fire[idx],
    feature_names=X_test.columns.tolist(),
    show=False
)
plt.title(f"Local explanation - test row #{idx}")
plt.tight_layout()
plt.savefig(OUT_DIR / "shap_local_explanation.png", dpi=150)
plt.close()
print(f"Saved local SHAP plot -> {OUT_DIR / 'shap_local_explanation.png'}")


# =========================================================================
# 5. Q4 - PLAIN-ENGLISH INTERPRETATION + ETHICS (dynamic, from real output)
# =========================================================================
top_feat = mean_abs_shap.index[0]
top_val = mean_abs_shap.iloc[0]

print("\n" + "=" * 74)
print("HUMANIZED SHAP INTERPRETATION (for your report / pitch video)")
print("=" * 74)
print(
    f"Across the whole test set, {top_feat} is the single feature that "
    f"moves the model's fire-risk prediction the most on average "
    f"(mean impact {top_val:.3f}). That lines up with the correlation "
    f"analysis from Q2 - the model is leaning on the same fire-danger "
    f"signal a human fire warden would, not on some opaque proxy.\n\n"
    f"For the individual test day we zoomed into (row #{idx}), the model "
    f"was {best_proba[idx]*100:.1f}% confident it was a fire day. The "
    f"waterfall plot shows exactly which of that day's readings pushed "
    f"the prediction up (toward 'fire') and which pulled it down - this "
    f"is the kind of explanation a fire-watch officer could actually "
    f"audit and challenge, rather than just trusting a black-box score."
)
print("=" * 74)

print("\n--- Ethics / responsible-use notes (Q4) ---")
print(
    "  - Bias/fairness : trained on 2 regions, 1 season -> do not assume it\n"
    "                    generalizes to other climates or years untested.\n"
    "  - Privacy        : public, weather-only data - low privacy risk, but\n"
    "                    real operational deployment should still follow\n"
    "                    local data-governance rules.\n"
    "  - Uncertainty    : treat predictions as decision SUPPORT, not an\n"
    "                    autonomous trigger for evacuation/resourcing.\n"
    "  - Cost asymmetry : a missed fire (false negative) is far costlier\n"
    "                    than a false alarm -> tune the decision threshold\n"
    "                    to favour recall, and review borderline cases.\n"
    "  - Human oversight: model should assist, not replace, fire-watch\n"
    "                    officers and emergency planners."
)

print("\nDone. All results saved to outputs/ - use model_comparison.csv, "
      "shap_global_importance.png and shap_local_explanation.png directly "
      "in your PPT and pitch video.")
